In [ ]:
import os, sys
import tempfile
from pathlib import Path
import pandas as pd
import norgatedata
import talib

from tqdm.auto import tqdm
from IPython.display import display
from collections import defaultdict
from typing import Tuple, List

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from alpha.engine.strategy import Strategy
from alpha.engine.backtest import run_daily
from alpha.engine.indicators import dv2_indicator

In [26]:
def build_index_constituent_matrix(indexname: str = 'S&P 500') -> Tuple[List[str], pd.DataFrame]:
    """
    builds a survivorship-bias-free universe matrix for backtesting.

    returns:
    - symbols: list of all symbols in the index.
    - dataframe with dates as index and tickers as columns.
        cells contain 1 if the stock was in the index on that date, 0 otherwise.
    """
    symbols = norgatedata.watchlist_symbols(f'{indexname} Current & Past')
    calendar = norgatedata.price_timeseries('$SPX', timeseriesformat='pandas-dataframe').index  # just to get the last trading day !

    last_trading_day = calendar[-1]
    universe_df = []

    for symbol in tqdm(symbols, desc='building universe'):
        idx = norgatedata.index_constituent_timeseries(symbol, indexname, timeseriesformat="pandas-dataframe")
        if idx['Index Constituent'].sum() > 0:  # checks if the symbol is in the index ('1') at that time
            idx = idx.rename(columns={'Index Constituent': symbol})
            idx = idx.loc[idx[symbol] == 1]
            # important: remove the past days as a safeguard
            if last_trading_day != idx.index[-1]:
                idx = idx.iloc[:-5]

            universe_df.append(idx)

    universe_df = pd.concat(universe_df, axis=1).fillna(0).astype(int).sort_index()

    return symbols, universe_df


def get_prices(symbols: List[str], benchmarks: List[str], start_date: str = '1998-01-01', end_date: str = None) -> pd.DataFrame:
    pricing_data = []

    # loop through all symbols (stocks + benchmarks)
    for symbol in tqdm(symbols + benchmarks, desc='downloading prices + features'):
        # use total return adjustment for benchmarks, capital/special adjustments for stocks
        if symbol in benchmarks:
            adjs = norgatedata.StockPriceAdjustmentType.TOTALRETURN  # only for benchmarks
        else:
            adjs = norgatedata.StockPriceAdjustmentType.CAPITALSPECIAL  # important for stocks, this prevents forward-looking bias (knowing divdends)

        # download price data with full market day padding from 1998 onward
        p = norgatedata.price_timeseries(
            symbol,
            stock_price_adjustment_setting=adjs,
            padding_setting=norgatedata.PaddingType.ALLMARKETDAYS,
            start_date=start_date,
            end_date=end_date,
            timeseriesformat='pandas-dataframe',
        )
        
        # skip if no data returned
        if len(p) == 0:
            continue

        # -------------------------------------------------- precomputing features -------------------------------------------------- #
        
        # For stock symbols only, compute features needed for the strategy
        if symbol in symbols:
            p['p126d_return'] = p['Close'] / p['Close'].shift(126) - 1
            p['natr'] = talib.NATR(p['High'], p['Low'], p['Close'], timeperiod=14)
            p['dv2'] = dv2_indicator(p['Close'], p['High'], p['Low'], length=126)
            p['sma_200'] = p['Close'].rolling(window=200).mean()

        # add a MultiIndex to columns to identify data by symbol
        p.columns = pd.MultiIndex.from_tuples([(symbol, c) for c in p.columns])
        pricing_data.append(p)

    # combine all pricing data into a single DataFrame, sorted by date
    pricing_data = pd.concat(pricing_data, axis=1).sort_index()

    return pricing_data


In [27]:
class DVO2Strategy(Strategy):
    max_positions = 10  # maximum number of positions to hold
    trade_id = 0  # intiliazing trade_id to 0
    current_trade = defaultdict(lambda: -1)  # initializing current_trade as a defaultdict of lists with default value -1
    universe_df = None  # df storing the universe of stocks

    def compute_signals(self, pricing_data: pd.DataFrame) -> pd.DataFrame:
        """
        this method is meant to generate trading signals but is currently 
        a placeholder. it will be implemented to process price data and 
        identify trade opportunities.
        """
        # in our case, we are already calculated this in the previous step during data ingestion
        return pricing_data  # currently returns raw pricing data (to be modified).

    def iterate(self, data: pd.DataFrame, close: pd.DataFrame, open_prices: pd.Series):
        """
        this method will contain the logic for executing trades based on 
        the strategy rules. it must be implemented to place buy and sell orders 
        according to QPI and price conditions.
        """

        # get current porfolio positions
        positions = self.get_positions()
        long_positions = positions[positions > 0]
        long_slots = self.max_positions - len(long_positions)  # calculate available slots for new positions

        # exit rules
        """ exit logic: sell if price > yesterday's high """
        for symbol in long_positions.index:
            c = close[(symbol, 'Close')]
            yh = data[(symbol, 'High')].iloc[-2]
            if c > yh:
                self.order_target_value(symbol, 0, trade_id=self.current_trade[symbol])
                long_slots += 1


        # entry rules
        capital_to_allocate_per_trade = self.previous_total_value / self.max_positions  # each trade receives an equal share of the portfolio
        long_opportunities = self.get_opportunities(close)

        while long_slots > 0 and len(long_opportunities) > 0:
            symbol = long_opportunities.pop(0)  # pips the top-ranked stock from opportunities

            # skip if already holding this stock (for safety)
            if self.get_position(symbol) != 0:
                continue

            # assign new trade_id and log it
            self.trade_id += 1
            self.current_trade[symbol] = self.trade_id

            # place buy order with equal capital allocation
            self.order_value(symbol, capital_to_allocate_per_trade, trade_id=self.trade_id)

            long_slots -= 1  # reduce available slots

    def get_opportunities(self, close) -> list:
        """ 
        identifying and ranking the best trade opportunities based on our entry criteria
        """
        # unstack multi-index DataFrame to have symbols as index and features as columns
        df = close.unstack().dropna()

        # remove benchmark symbols (e.g., $SPX) from the dataset
        df = df[~df.index.astype(str).str.startswith('$')]

        # apply entry filters:
        # - dv2<10
        # - Close[0] > sma(200) (uptrend condition)
        # - Return over 126 days > 0 (positive long-term momentum)
        # then sort by NATR in descending order for liquidity ranking
        df = df[
            (df['dv2'] < 10) &
            (df['Close'] > df['sma_200']) &
            (df['p126d_return'] > 0)
        ].sort_values('natr', ascending=False)

        # get the list of stocks in the universe on the previous trading day
        u = self.universe_df.loc[self.previous_bar]
        u = u[u == 1].index.tolist()

        # return the filtered list of opportunity tickers that are also in the universe
        return df[df.index.isin(u)].index.tolist()


In [ ]:
benchmarks = ['$SPX']
index_symbols, universe_df = build_index_constituent_matrix(indexname='S&P 500')
pricing_data = get_prices(index_symbols, benchmarks, start_date='1998-01-01', end_date=None)
...
sa = DVO2Strategy(name='DVO2Strategy', benchmarks=benchmarks, capital_base=100_000, slippage=0.0001)
sa.universe_df = universe_df
calendar = pricing_data.index
calendar = calendar[calendar.year >= 2004]

run_daily(sa, pricing_data, calendar)

sa.universe_df = None

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
display(sa.summary)
display(sa.summary_trades)
tmp = Path(tempfile.gettempdir())
sa.to_pickle(tmp / f'{sa.name}.pkl')
sa.plot('$SPX', 'S&P 500', save_to=tmp / f'{sa.name}.png')

In [29]:
# genereate a monthly report
from alpha.engine.metrics import generate_monthly_returns
mr = generate_monthly_returns(sa.results['total_value'], add_sharpe_ratios=True, add_max_drawdowns=True)
for c in mr.columns[:-1]:
    mr[c] = mr[c].apply(lambda x: f'{x:.1%}' if not pd.isna(x) else "")
mr['Sharpe Ratio'] = mr['Sharpe Ratio'].apply(lambda x: f'{x:.2f}' if not pd.isna(x) else "")
display(mr.style.background_gradient(cmap='coolwarm'))


month,1,2,3,4,5,6,7,8,9,10,11,12,Annual Return,Max Drawdown,Sharpe Ratio
year,,,,,,,,,,,,,,,
2025,8.2%,-1.9%,-0.2%,0.3%,-0.3%,,,,,,,,5.9%,-11.7%,0.70
2024,1.6%,4.7%,-0.8%,-0.5%,5.7%,2.6%,1.4%,-4.1%,3.1%,-5.7%,8.8%,-7.8%,8.1%,-16.8%,0.54
2023,0.7%,2.8%,-6.2%,3.0%,-3.0%,10.4%,6.8%,-0.5%,-2.9%,-7.8%,6.3%,3.9%,12.6%,-13.6%,0.89
2022,-11.7%,3.2%,4.7%,-8.8%,5.4%,-4.5%,2.4%,0.1%,-0.7%,10.4%,6.8%,-0.5%,4.6%,-18.6%,0.32
2021,-1.3%,10.9%,5.4%,4.7%,-1.8%,-0.1%,-0.5%,1.1%,-3.7%,3.0%,-2.6%,4.9%,20.8%,-9.4%,1.06
2020,-3.8%,-2.7%,-9.6%,28.3%,5.1%,-1.8%,3.5%,2.3%,0.8%,0.5%,34.5%,6.7%,72.6%,-29.6%,1.86
2019,2.8%,2.2%,-0.3%,-0.3%,-7.6%,5.8%,-3.1%,-2.3%,3.0%,3.3%,5.5%,3.6%,12.6%,-11.8%,0.93
2018,8.1%,-3.9%,-1.7%,3.1%,7.6%,-0.5%,3.6%,2.1%,4.2%,-5.9%,3.2%,-6.2%,13.3%,-13.3%,0.75
2017,4.7%,1.3%,-5.4%,-0.3%,-1.2%,-3.3%,3.1%,-0.7%,1.6%,0.0%,5.4%,4.3%,9.2%,-12.1%,0.78
